# Synthetic Impacts and Risk Metrics

This notebook demonstrates how to generate synthetic flood events using the calibrated EVT model, convert them into flood impacts using the CLIMADA hazard module and population exposure, and compute risk metrics such as the AEP/OEP curves and the Annual Average People Affected (AAPA).

* **Inputs**: Fitted EVT model parameters, basin configuration, vulnerability assumptions.
* **Outputs**: Synthetic impact database, AEP/OEP curves, AAPA values and proposed trigger levels.

Follow the steps below to load your calibrated parameters, simulate synthetic events, build hazards using CLIMADA, compute impacts and derive risk metrics.

In [ ]:
from philflood.basin import BasinConfig
from philflood.ev.synthetic_events import simulate_multiple_years
from philflood.ev.gpd_fit import GPDModel
from philflood.hazard.climada_driver import run_flood_hazard
from philflood.impact.population_exposure import load_population_grid
from philflood.impact.vulnerability import compute_people_affected
from philflood.risk.aep_oep import compute_aep_oep, compute_aapa
from philflood.risk.trigger_design import propose_candidate_triggers

# TODO: replace with the basin's calibrated parameters
model = GPDModel(threshold=1.0, shape=0.0, scale=0.5, rate=2.0)

# Simulate synthetic years of discharge peaks
synthetic_events = simulate_multiple_years(model, years=100, basin_id='example_basin')
print(synthetic_events.head())

# TODO: convert discharge to hazard using run_flood_hazard and then to impacts
# For now we skip to computing AEP/OEP on random impacts
import pandas as pd
synthetic_events['year'] = synthetic_events['year']
synthetic_events['people_affected'] = (synthetic_events['discharge'] * 10000).clip(0)

aep_curve, oep_curve = compute_aep_oep(synthetic_events, value_col='people_affected', year_col='year')
aapa_value = compute_aapa(synthetic_events, value_col='people_affected', year_col='year')

print('AAPA:', aapa_value)
print(aep_curve.head())
print(oep_curve.head())

# Propose some candidate trigger return periods
triggers = propose_candidate_triggers(aep_curve, return_periods=[5, 10, 20])
print('Candidate triggers:', triggers)

# TODO: save results to config files or pass to the next notebook
